# Email Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("suvroo/email-sentiment-analysis-dataset")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

In [ ]:
ed1=pd.read_csv(path+"/Email Analysis Dataset.csv")
display(ed1.head())
ed2=pd.read_csv(path+"/contacts - dimension table.csv")
display(ed2.head())

In [ ]:
ed1.columns

In [ ]:
ed2.columns

In [ ]:
ed1.info()
ed2.info()

In [ ]:
display(ed1.describe())
display(ed2.describe())

In [ ]:
display(ed1.isnull().sum())
display(ed2.isnull().sum())

## Convert Date Properly

In [ ]:
ed1['Date'] = pd.to_datetime(ed1['Date'])
ed1


In [ ]:
ed1.isnull().sum()


## Remove Non-Useful Columns

In [ ]:
ed1 = ed1.drop(columns=[
    'Email id'
])

## Fix Column Names (clean formatting)

In [ ]:
ed1.columns = ed1.columns.str.strip().str.lower().str.replace(' ', '_')
ed1.head()


In [ ]:
ed1.duplicated().sum()
ed1 = ed1.drop_duplicates()

In [ ]:
ed1.info()

# Plots using ggplot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

## Sentiment Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='sentiment', data=ed1)
plt.title("Email Sentiment Distribution")
plt.show()


## Emails by Department

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(y='from_department', data=ed1,
              order=ed1['from_department'].value_counts().index)
plt.title("Emails Sent by Department")
plt.show()


## Device Usage

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='device', data=ed1)
plt.title("Device Used for Emails")
plt.show()

## Emails Within Work Hours

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='within_work_hours', data=ed1)
plt.title("Emails Sent Within Work Hours")
plt.show()

## Sentiment vs Email Opened

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='is_opened?', hue='sentiment', data=ed1)
plt.title("Email Open Status vs Sentiment")
plt.show()


# Feature Engineering

## Time Features from Date

In [ ]:
ed1['year'] = ed1['date'].dt.year
ed1['month'] = ed1['date'].dt.month
ed1['day'] = ed1['date'].dt.day
ed1['dayofweek'] = ed1['date'].dt.dayofweek


## Convert Yes/No Columns to Binary

In [ ]:
binary_cols = ['is_opened?', 'within_work_hours', 'within_workdays']

for col in binary_cols:
    ed1[col] = ed1[col].map({'Yes':1, 'No':0})


## Target Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

sentiment_encoder = LabelEncoder()
ed1['sentiment_label'] = sentiment_encoder.fit_transform(ed1['sentiment'])


## Remove Non-Learning Columns

In [ ]:
ed1 = ed1.drop(columns=[
    'from_name',
    'to_name',
    'date',
    'sentiment'
])


## Encode Remaining Categories

In [ ]:
from sklearn.preprocessing import LabelEncoder

for col in ed1.columns:
    if ed1[col].dtype == 'object':
        le = LabelEncoder()
        ed1[col] = le.fit_transform(ed1[col])


# Scaling

In [ ]:
X = ed1.drop('sentiment_label', axis=1)
y = ed1['sentiment_label']

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X = pd.DataFrame(X_scaled, columns=X.columns)

# Train Test  Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)


# Model Training

## RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
email_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

In [ ]:
email_model.fit(X_train, y_train)

# Predictions

In [ ]:
y_pred = email_model.predict(X_test)
y_pred

In [ ]:
y_prob = email_model.predict_proba(X_test)
y_prob.max()

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

results.head()


# Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


#

## XGBoost (Better for Patterns)

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)


In [ ]:
xgb_pred

#Evaluations

In [ ]:
print("XGBoost Accuracy:",
      accuracy_score(y_test, xgb_pred))

print(classification_report(y_test, xgb_pred))


# conclusion

sentiment mainly comes from WHAT is written in the email, not only metadata.

So without NLP, model is almost guessing.

# Handle Class Imbalance

### Apply SMOTE balancing

In [ ]:
!pip install imbalanced-learn


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)


In [ ]:
xgb_model.fit(X_train_sm, y_train_sm)


In [ ]:
y_pred = xgb_model.predict(X_test)


In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix After SMOTE")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
print("Accuracy before SMOTE: 0.48")
print("Accuracy after SMOTE:", accuracy_score(y_test, y_pred))
